# Phase 1 — raw source inspection

Statistics of the five sources **as published**, before any filtering, pairing or
randomization by our pipeline. The logic lives in `prepare_data/inspect_raw.py` so
that it is rerunnable and testable; this notebook is the interactive view of it.

Run `python -m prepare_data.download --source all` first.

In [ ]:
import os, sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from prepare_data import inspect_raw
from prepare_data.config import ALL_SOURCES
from prepare_data.download import read_log

ALL_SOURCES

## What arrived, and what it cost

Download size and wall time are the evidence for the bottleneck question, so they
are read back from the logs rather than recalled.

In [ ]:
import pandas as pd

logs = {s: (read_log(s) or {"status": "not attempted"}) for s in ALL_SOURCES}
pd.DataFrame(logs).T[["status", "bytes", "seconds"]]

## Per-source statistics

Row counts, field names, label value counts, winner-position balance, annotator-id
presence, response-length distribution and how often the longer response wins.

In [ ]:
results = inspect_raw.main(sources=ALL_SOURCES)

for source, result in results.items():
    print(f"\n{'=' * 70}\n{source}\n{'=' * 70}")
    if "error" in result:
        print("not inspected:", result["error"])
        continue
    for key, value in result.items():
        print(f"  {key}: {value}")

## Label skew at a glance

`p_longer_is_preferred` far from 0.5 means a judge can score well on that source by
preferring length alone, independently of anything visual.

In [ ]:
rows = [
    {
        "source": source,
        "rows": result.get("rows"),
        "annotator_ids": result.get("annotator_ids"),
        "winner_position_balance": result.get("winner_position_balance"),
        "p_longer_is_preferred": result.get("p_longer_is_preferred"),
    }
    for source, result in results.items()
    if "error" not in result
]
pd.DataFrame(rows).set_index("source")

The full write-up lands in `reports/raw_inspection.md`. Phase 2-5 then run with

```bash
python -m prepare_data.run --source all
```

and the post-filtering numbers appear in `reports/bias_tables.md`.